# Resource estimation for simulating a 2D Ising Model Hamiltonian

In this Python notebook we demonstrate how to estimate the resources for quantum dynamics, specifically the simulation of an Ising model Hamiltonian on an $N \times N$ 2D
lattice using a *fourth-order Trotter Suzuki product formula* assuming a 2D
qubit architecture with nearest-neighbor connectivity.

First, we load the qsharp package.

In [1]:
import qsharp

## Background: 2D Ising model

The Ising model is a mathematical model of ferromagnetism in a lattice (in our case a 2D square lattice) with two kinds of terms in the Hamiltonian: (i) an interaction term between adjacent sites and (ii) an external magnetic field acting at each site. For our purposes, we consider a simplified version of the model where the interaction terms have the same strength and the external field strength is the same at each site.
Formally, the Ising model Hamiltonian on an $N \times N$ lattice we consider is formulated as:

$$
H = \underbrace{-J \sum_{i, j} Z_i Z_j}_{B} - \underbrace{h \sum_j X_j}_{A}
$$
where $J$ is the interaction strength, $h$ is external field strength.

In [2]:
from qsharp.magnets.geometry import Patch2D
from qsharp.magnets.models import IsingModel

model = IsingModel(Patch2D(10, 10, self_loops=True), h=0.5, J=1.0)
print (model)

Ising model with 2 terms on 100 qubits (h=0.5, J=1.0).


The time evolution $e^{-iHt}$ for the Hamiltonian is simulated with the fourth-order product formula so that any errors in simulation are sufficiently small. Essentially, this is done by simulating the evolution for small slices of time $\Delta$ and repeating this for `nSteps` $= t/\Delta$ to obtain the full time evolution. The Trotter-Suzuki formula for higher orders can be recursively defined using a *fractal decomposition* as discussed in Section 3 of [Hatanao and Suziki's survey](https://link.springer.com/chapter/10.1007/11526216_2). Then the fourth order formula $U_4(\Delta)$ can be constructed using the second-order one $U_2(\Delta)$ as follows.
$$
\begin{aligned}
U_2(\Delta) & = e^{-iA\Delta/2} e^{-iB\Delta} e^{-iA\Delta/2}; \\
U_4(\Delta) & = U_2(p\Delta)U_2(p\Delta)U_2((1 - 4p)\Delta)U_2(p\Delta)U_2(p\Delta); \\
p & = (4 - 4^{1/3})^{-1}.
\end{aligned}
$$

In [3]:
import math

from qsharp.magnets.trotter import TrotterExpansion, fourth_order_trotter_suzuki

T = 10
dt = 0.5

trotter = TrotterExpansion(fourth_order_trotter_suzuki, model, time=dt, num_steps=math.ceil(T/dt))
circuit = trotter.cirq()
print (circuit)

[                      ┌─────────────────────────────────┐   ┌────────────────────────────────────────────────────────────────────────────────────────┐                 ┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────┐                 ┌─────────────────────────────────┐   ┌────────────────────────────────────────────────────────────────────────────────────────┐                 ┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────┐                  ┌────────────────────────────────────┐   ┌────────────────────────────────────────────────────────────────────────────────────────────────┐                  ┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐                 ┌─────────────────────────────────┐   ┌────────────────────────────────────────────────────────────────────────────────────────┐    

## Application wrapper

In [4]:
from qsharp.qre.application import CirqApplication

app = CirqApplication(circuit)
app_query = app.q()
print (app)

CirqApplication(circuit_or_qasm=cirq.CircuitOperation(
    circuit=cirq.FrozenCircuit([
        cirq.Moment(
            cirq.PauliStringPhasor(((1+0j)*cirq.X(cirq.LineQubit(18))), qubits=(cirq.LineQubit(18),), exponent_neg=0.0014843817645604966, exponent_pos=0),
            cirq.PauliStringPhasor(((1+0j)*cirq.X(cirq.LineQubit(19))), qubits=(cirq.LineQubit(19),), exponent_neg=0.0014843817645604966, exponent_pos=0),
            cirq.PauliStringPhasor(((1+0j)*cirq.X(cirq.LineQubit(14))), qubits=(cirq.LineQubit(14),), exponent_neg=0.0014843817645604966, exponent_pos=0),
            cirq.PauliStringPhasor(((1+0j)*cirq.X(cirq.LineQubit(17))), qubits=(cirq.LineQubit(17),), exponent_neg=0.0014843817645604966, exponent_pos=0),
            cirq.PauliStringPhasor(((1+0j)*cirq.X(cirq.LineQubit(28))), qubits=(cirq.LineQubit(28),), exponent_neg=0.0014843817645604966, exponent_pos=0),
            cirq.PauliStringPhasor(((1+0j)*cirq.X(cirq.LineQubit(20))), qubits=(cirq.LineQubit(20),), exponent_neg=0

## Majorana architectures


In [5]:
from qsharp.qre.models import Majorana

arch = Majorana(error_rate=1e-5)
print (arch)

Majorana(error_rate=1e-05)


## Creating Application and ISA Queries 

In [6]:
from qsharp.qre import estimate, PSSPC, LatticeSurgery
from qsharp.qre.models import ThreeAux, RoundBasedFactory

trace_query = (
    app_query
    * PSSPC.q(num_ts_per_rotation=[15,16,17,18])
    * LatticeSurgery.q()
)

isa_query = (
    ThreeAux.q(distance=[5,7,11,13,15,17])
    * RoundBasedFactory.q(code_query=ThreeAux.q())
)



## Visualizing and understanding the results

### Result summary table

In [7]:
from qsharp.qre.instruction_ids import LATTICE_SURGERY
from qsharp.qre.property_keys import NUM_TS_PER_ROTATION, DISTANCE, PHYSICAL_COMPUTE_QUBITS

results = estimate(app, arch, isa_query, trace_query, max_error=0.01, name="Majorana e-5, 3-aux")

results.add_column("compute_distance", lambda entry: entry.source[LATTICE_SURGERY].instruction[DISTANCE])
results.add_column("compute qubits", lambda entry: entry.properties[PHYSICAL_COMPUTE_QUBITS])
results.add_column("num_ts_per_rotation", lambda entry: entry.properties[NUM_TS_PER_ROTATION])
results.add_factory_summary_column()
results.add_column("cycle_time", lambda entry: entry.source[LATTICE_SURGERY].instruction.expect_time(1))

print (results.as_frame())

                  name  qubits                runtime     error  \
0  Majorana e-5, 3-aux  332502 0 days 00:00:01.864320  0.004568   
1  Majorana e-5, 3-aux  377198 0 days 00:00:01.242880  0.004571   
2  Majorana e-5, 3-aux  466054 0 days 00:00:00.932160  0.006122   

   compute_distance  compute qubits  num_ts_per_rotation factories  cycle_time  
0                11          110630                   17      98×T       48000  
1                 7           44390                   17     147×T       32000  
2                 5           22310                   17     196×T       24000  


## Throttling the algorithm

In [9]:
new_trace_query = (
    app_query
    * PSSPC.q(slow_down_factor = 3.0, num_ts_per_rotation=[15,16,17,18])
    * LatticeSurgery.q(slow_down_factor = 5.0)
)

new_results = estimate(app, arch, isa_query, new_trace_query, max_error=0.01, name="Majorana e-5, 3-aux")
new_results.add_column("compute_distance", lambda entry: entry.source[LATTICE_SURGERY].instruction[DISTANCE])
new_results.add_column("compute qubits", lambda entry: entry.properties[PHYSICAL_COMPUTE_QUBITS])
new_results.add_column("num_ts_per_rotation", lambda entry: entry.properties[NUM_TS_PER_ROTATION])
new_results.add_factory_summary_column()
new_results.add_column("cycle_time", lambda entry: entry.source[LATTICE_SURGERY].instruction.expect_time(1))

print (new_results.as_frame())

                  name  qubits                runtime     error  \
0  Majorana e-5, 3-aux  112310 0 days 00:00:06.214400  0.004580   
1  Majorana e-5, 3-aux  115134 0 days 00:00:04.723200  0.009699   

   compute_distance  compute qubits  num_ts_per_rotation factories  cycle_time  
0                 7           44390                   17      30×T       32000  
1                 5           22310                   18      41×T       24000  


## Next steps

Feel free to use this notebook as a starting point for your own experiments.  For
example, you can

* explore how the results change considering other problem instances of the Heisenberg model
* explore space- and time-trade-offs by changing the value for
  `logical_depth_factor` or `max_t_factories`
* visualize these trade-offs with the space and time diagrams
* use other or customized qubit parameters